# Support Vector Machine Modeling Across Healthcare Datasets

This notebook applies Support Vector Machine methods across multiple healthcare-related datasets, including diabetes, life expectancy, and SUPPORT2. The workflow includes dataset loading, train-test splitting, feature scaling or preprocessing, hyperparameter tuning, and model evaluation.

The diabetes and SUPPORT2 datasets are treated as classification tasks using Support Vector Classifiers, while the life expectancy dataset is treated as a regression task using Support Vector Regression. The goal is to compare how SVM-based methods perform across different healthcare prediction problems and evaluate results using appropriate metrics for each task.

This notebook is included in the modeling folder because the main focus is supervised model development, kernel selection, regularization, hyperparameter comparison, and performance evaluation rather than exploratory data analysis.


## Diabetes Dataset: Support Vector Classification

This section trains a regularized Support Vector Classifier on the diabetes health indicators dataset. The model uses scaling, class balancing, and grid search to compare linear and RBF kernels, with performance evaluated using accuracy, balanced accuracy, and macro F1-score.


In [2]:
import pandas as pd

diabetes_df = pd.read_csv("diabetes_012_health_indicators_BRFSS2015.csv")

In [3]:
#Fast, regularized SVM on diabetes_df
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

target = "Diabetes_012"       
max_n  = 15000                   # cap rows to keep runtime short
rs     = 42

df = diabetes_df.dropna(subset=[target]).drop_duplicates()
X, y = df.drop(columns=[target]), df[target]

# stratified subsample for speed
if len(df) > max_n:
    X, _, y, _ = train_test_split(X, y, train_size=max_n, stratify=y, random_state=rs)

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=rs)

pipe = Pipeline([("scaler", StandardScaler()), ("svm", SVC(class_weight="balanced"))])

param_grid = [
    {"svm__kernel": ["linear"], "svm__C": [0.1, 1, 10]},                    
    {"svm__kernel": ["rbf"],    "svm__C": [0.5, 1, 2], "svm__gamma": ["scale", 0.1]}, 
]

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=rs)
gs = GridSearchCV(pipe, param_grid, scoring="f1_macro", cv=cv, n_jobs=-1, verbose=0)
gs.fit(Xtr, ytr)

pred = gs.best_estimator_.predict(Xte)
print("Best params:", gs.best_params_)
print(f"Accuracy: {accuracy_score(yte, pred):.3f} | Balanced Acc: {balanced_accuracy_score(yte, pred):.3f} | F1(macro): {f1_score(yte, pred, average='macro'):.3f}")

Best params: {'svm__C': 1, 'svm__gamma': 0.1, 'svm__kernel': 'rbf'}
Accuracy: 0.685 | Balanced Acc: 0.465 | F1(macro): 0.420


**Diabetes modeling summary:** The best diabetes model used an RBF kernel. The results show moderate overall accuracy, while balanced accuracy and macro F1 indicate that class imbalance remains a challenge for predicting all diabetes outcome groups equally.


## Life Expectancy Dataset: Support Vector Regression

This section trains a Support Vector Regression model on the imputed life expectancy dataset. Numeric and categorical predictors are preprocessed before comparing linear and RBF kernels through grid search. Model performance is evaluated using RMSE, MAE, and R².


In [5]:
lifeexp_df  = pd.read_csv("Life Expectancy Data - mean_mode_imputed.csv")

In [6]:
#SVR (SVM regression) on lifeexp_df with ordinal encoding for fast and regularized 
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

MAX_N, RS = 15000, 42

# Pick target 
cands = [c for c in lifeexp_df.columns if ("life" in c.lower() and "expect" in c.lower())]
TARGET = cands[0] if cands else "Life expectancy"

df = lifeexp_df.dropna(subset=[TARGET]).drop_duplicates().copy()
y = pd.to_numeric(df[TARGET], errors="coerce")
X = df.drop(columns=[TARGET])

# Columns by type
num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = X.select_dtypes(exclude="number").columns.tolist()

#collapse rare categories to "Other" for stability
for c in cat_cols:
    counts = X[c].fillna("MISSING").value_counts()
    rare = counts[counts < 10].index
    X[c] = X[c].fillna("MISSING").mask(X[c].isin(rare), "Other")

# Subsample for speed
if len(X) > MAX_N:
    X, _, y, _ = train_test_split(X, y, train_size=MAX_N, random_state=RS)

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=RS)

# Preprocessing: impute+scale numerics, impute+ordinal-encode categoricals
pre = ColumnTransformer([
    ("num", Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc", StandardScaler()),
    ]), num_cols),
    ("cat", Pipeline([
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("ord", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
    ]), cat_cols),
], remainder="drop")

pipe = Pipeline([("pre", pre), ("svr", SVR())])

# Tiny grid: regularization via C; kernel trick via RBF with gamma
param_grid = [
    {"svr__kernel": ["linear"], "svr__C": [0.5, 1, 2]},
    {"svr__kernel": ["rbf"],    "svr__C": [0.5, 1, 2], "svr__gamma": ["scale", 0.1]},
]

cv = KFold(n_splits=3, shuffle=True, random_state=RS)
gs = GridSearchCV(pipe, param_grid, scoring="neg_root_mean_squared_error", cv=cv, n_jobs=-1)
gs.fit(Xtr, ytr)

#Metrics
pred = gs.best_estimator_.predict(Xte)
mse  = mean_squared_error(yte, pred)     
rmse = np.sqrt(mse)
mae  = mean_absolute_error(yte, pred)
r2   = r2_score(yte, pred)

print("Best params:", gs.best_params_)
print(f"RMSE: {rmse:.3f} | MAE: {mae:.3f} | R^2: {r2:.3f}")

Best params: {'svr__C': 2, 'svr__kernel': 'linear'}
RMSE: 4.126 | MAE: 2.906 | R^2: 0.803


**Life expectancy modeling summary:** The best SVR model used a linear kernel and achieved strong predictive performance, suggesting that the preprocessing and regularized regression approach captured much of the signal in the life expectancy dataset.


## SUPPORT2 Dataset: Support Vector Classification

This section trains a Support Vector Machine model on the imputed SUPPORT2 dataset using hospital death as the target variable. The workflow handles numeric and categorical predictors, applies preprocessing, compares SVM kernel settings, and evaluates classification performance using accuracy, balanced accuracy, and macro F1-score.


In [8]:
support2_df = pd.read_csv("support2_imputed.csv")

In [9]:
#SVM on support2_df (ordinal encoding)
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.svm import SVC, SVR
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, mean_squared_error, mean_absolute_error, r2_score

target = "hospdead"       
max_n, rs = 15000, 42

df = support2_df.dropna(subset=[target]).drop_duplicates().copy()
y  = df[target];  X = df.drop(columns=[target])

def int_like(s): 
    s = pd.to_numeric(s, errors="coerce"); return np.all(np.isfinite(s)) and np.allclose(s, s.astype(int))
is_cls = (y.nunique() <= 10) and int_like(y)

# split columns
num_cols = X.select_dtypes(include="number").columns
cat_cols = X.select_dtypes(exclude="number").columns

# subsample for speed
if len(X) > max_n:
    strat = y if is_cls else None
    X, _, y, _ = train_test_split(X, y, train_size=max_n, random_state=rs, stratify=strat)

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=rs, stratify=y if is_cls else None)

pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num_cols),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("enc", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))]), cat_cols)
])

if is_cls:
    pipe = Pipeline([("pre", pre), ("svm", SVC(class_weight="balanced"))])
    grid = [{"svm__kernel": ["linear"], "svm__C": [0.1, 1, 10]},
            {"svm__kernel": ["rbf"],    "svm__C": [0.5, 1, 2], "svm__gamma": ["scale", 0.1]}]
    gs = GridSearchCV(pipe, grid, scoring="f1_macro", cv=StratifiedKFold(3, shuffle=True, random_state=rs), n_jobs=-1)
    gs.fit(Xtr, ytr)
    p = gs.best_estimator_.predict(Xte)
    print("Task: classification |", "Best:", gs.best_params_)
    print(f"Acc: {accuracy_score(yte, p):.3f} | BalAcc: {balanced_accuracy_score(yte, p):.3f} | F1(macro): {f1_score(yte, p, average='macro'):.3f}")
else:
    ytr = pd.to_numeric(ytr, errors="coerce"); yte = pd.to_numeric(yte, errors="coerce")
    pipe = Pipeline([("pre", pre), ("svr", SVR())])
    grid = [{"svr__kernel": ["linear"], "svr__C": [0.5, 1, 2]},
            {"svr__kernel": ["rbf"],    "svr__C": [0.5, 1, 2], "svr__gamma": ["scale", 0.1]}]
    gs = GridSearchCV(pipe, grid, scoring="neg_mean_squared_error", cv=KFold(3, shuffle=True, random_state=rs), n_jobs=-1)
    gs.fit(Xtr, ytr)
    p = gs.best_estimator_.predict(Xte)
    rmse = np.sqrt(mean_squared_error(yte, p)); mae = mean_absolute_error(yte, p); r2 = r2_score(yte, p)
    print("Task: regression     |", "Best:", gs.best_params_)
    print(f"RMSE: {rmse:.3f} | MAE: {mae:.3f} | R^2: {r2:.3f}")

Task: classification | Best: {'svm__C': 0.1, 'svm__kernel': 'linear'}
Acc: 0.910 | BalAcc: 0.923 | F1(macro): 0.891


**SUPPORT2 modeling summary:** The SUPPORT2 model achieved strong classification performance, with high accuracy, balanced accuracy, and macro F1-score. These results suggest that the SVM approach was effective for this binary clinical outcome in the prepared dataset.
